# Fast-fail verdict — AE trivial inversion (latent recovery)

**Question**: can the foundation model's latent space be navigated by gradient
optimization well enough to represent true velocity models? (the
reparameterized-FWI premise)

**Verdict — DRAFT (rewrite as you see fit)**:

- **Reachability**: from every latent init, inversion converges to
  ~0.0043–0.0047 final MSE at the best configs — the manifold is reachable
  without a supervised warm start.
- **Gap vs supervised**: on the same (difficulty-matched) examples, latent
  recovery lands within a small factor of the refit model's prediction.
- **Caveat**: difficulty is example-driven — the same examples stay
  hardest for every init and every ML model.


## Search space

| dimension | values |
|---|---|
| refit ML models (4) | encdec 3×3 (seed 7) · encdec 5×5 (seed 79) · encdec 7×7 (seed 571) · transencdec 5×5 (seed 571) |
| latent inits (4) | `trained` (latent of a randomly drawn training example — seed-coupled) · `zeros` · `random` (U[0,1)) · `randomn` (N(0,1)) |
| learning rates (5) | 0.001 · 0.01 · 0.1 · 0.5 · 1.0 |
| epoch budgets (4) | 50 · 100 · 500 · 1000 |
| target examples | the same 3 per ML model: holdout positions 1, 10, 20 — difficulty-matched throughout |
| clean runs | 4 × 4 × 5 × 4 = **320** (complete factorial; active MLflow runs only) |

Most figures use the **best ML model** — the refit with the lowest holdout
prediction MSE (selected in the next cell, not hardcoded).


In [ ]:
# minimal data load (provenance: notebooks/data/DATA_DICTIONARY.md;
# extraction: notebooks/data/extract_from_mlflow.py, ACTIVE runs only)
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from matplotlib.patches import Rectangle

DATA = Path("data") if Path("data/runs.csv").exists() else Path("notebooks/data")
runs = pd.read_csv(DATA / "runs.csv", parse_dates=["start_time_iso"])
kept = runs[(runs["group"] == "sweep_full_samples")
            & (runs["kept_in_clean_set"] == True)].copy()  # noqa: E712
per_model = pd.read_csv(DATA / "inversion_per_model.csv")
per_model = per_model[per_model["run_id"].isin(kept["run_id"])].copy()
hpe = pd.read_csv(DATA / "train_loss_per_example.csv")
inv_ex = pd.read_csv(DATA / "inverted_examples.csv")
base = pd.read_csv(DATA / "refit_baselines.csv").set_index("refit_short")
assert len(kept) == 320 and kept["combo"].nunique() == 320

INITS = ["trained", "zeros", "random", "randomn"]
LRS = [0.001, 0.01, 0.1, 0.5, 1.0]
BUDGETS = [50, 100, 500, 1000]
EX_POS = [1, 10, 20]

# best ML model = lowest refit prediction MSE on its 100-example holdout
BEST = base["mse_mean_test_set"].idxmin()
labels = kept.drop_duplicates("refit_short").set_index("refit_short")
row = labels.loc[BEST]
ex_name = {int(r.example_pos): int(r.dataset_example_idx)
           for r in inv_ex[inv_ex.refit_short == BEST].itertuples()}
print(f"analysis set: {len(kept)} clean runs | best ML model: "
      f"{row.arch_family} {int(row.kernel)}x{int(row.kernel)}, seed {int(row.seed)} "
      f"({BEST}, holdout prediction MSE {base.loc[BEST, 'mse_mean_test_set']:.5f})")


In [ ]:
# FIGURE 1 — experiment grid at the best ML model: final-MSE landscape
# per budget (init x lr); color = mean over the 3 inverted examples.
# magma (perceptual; black = best/low MSE) + PowerNorm(gamma=0.5) for
# low-range emphasis; PER-BUDGET min/max scale; best cell outlined.
sub = kept[kept["refit_short"] == BEST]
grid = (sub.groupby(["latent_init", "lr", "max_epochs"], as_index=False)
        ["inversion_mse_mean_test_set"].mean())
fig, axes = plt.subplots(1, 4, figsize=(21, 4.5), sharey=True)
for ax, ep in zip(axes, BUDGETS):
    M = (grid[grid["max_epochs"] == ep]
         .pivot(index="latent_init", columns="lr",
                values="inversion_mse_mean_test_set")
         .reindex(index=INITS, columns=LRS))
    vmin, vmax = M.values.min(), M.values.max()   # per-budget scale
    ax.imshow(M.values, cmap="magma",
              norm=PowerNorm(gamma=0.5, vmin=vmin, vmax=vmax), aspect="auto")
    ax.set_xticks(range(len(LRS))); ax.set_xticklabels(LRS, rotation=45, fontsize=8)
    ax.set_yticks(range(len(INITS))); ax.set_yticklabels(INITS, fontsize=9)
    bi, bj = divmod(M.values.argmin(), M.values.shape[1])
    ax.add_patch(Rectangle((bj - 0.5, bi - 0.5), 1, 1, fill=False,
                           edgecolor="cyan", lw=2))
    for i in range(len(INITS)):
        for j in range(len(LRS)):
            nrm = ((M.values[i, j] - vmin) / (vmax - vmin)) ** 0.5
            ax.text(j, i, f"{M.values[i, j]:.4f}", ha="center", va="center",
                    fontsize=7, color="white" if nrm < 0.45 else "black")
    ax.set_title(f"budget = {ep} epochs (own scale)", fontsize=10)
    print(f"best @ {ep}ep: {M.index[bi]} lr={M.columns[bj]} "
          f"({M.values[bi, bj]:.4f})")
fig.suptitle("experiment grid — best ML model (encdec 5x5, seed 79), "
             "mean over the 3 inverted examples (darker = better; best cell outlined)",
             y=1.06)
plt.tight_layout()
plt.show()


In [ ]:
# FIGURE 2 — convergence, PER EXAMPLE (best ML model; inits trained & random).
# y-axis = the INVERSION loss (the latent-optimization objective: model-space
# MSE between Generator(z) and the target example) — NOT the refit training
# loss. It is logged under the metric name 'train_loss' because the inversion
# reuses the skorch training machinery to train the latent. One trajectory
# per (init, target example); each init at ITS best config (same selection
# rule as every other figure: lowest mean final MSE over the 3 targets).
SHOW_INITS = ["trained", "random"]
sub = kept[kept["refit_short"] == BEST]
best_cfg = sub.loc[sub.groupby("latent_init")["inversion_mse_mean_test_set"].idxmin()]
best_cfg = best_cfg.set_index("latent_init")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, mi in zip(axes, EX_POS):
    for init in SHOW_INITS:
        cfg = best_cfg.loc[init]
        run = kept[(kept["refit_short"] == BEST) & (kept["latent_init"] == init)
                   & (kept["lr"] == cfg["lr"])
                   & (kept["max_epochs"] == cfg["max_epochs"])]
        g = (hpe[hpe["run_id"].isin(run["run_id"]) & (hpe["model_idx"] == mi)]
             .groupby("step", as_index=False)["value"].last().sort_values("step"))
        ax.plot(g["step"], g["value"], lw=1.2, color=f"C{INITS.index(init)}",
                label=f"{init} (lr={cfg['lr']}, {int(cfg['max_epochs'])}ep)")
    ax.set_yscale("log")
    ax.set_xlabel("epoch step")
    ax.set_title(f"target — pos {mi} (example {ex_name[mi]})", fontsize=10)
axes[0].set_ylabel("inversion loss\n(model-space MSE)")
axes[0].legend(fontsize=8)
fig.suptitle("convergence per example — best ML model, inits trained & random "
             "at their best configs (shared y-scale: compare heights across panels)",
             y=1.05)
plt.tight_layout()
plt.show()
